# optimizer-class-dispatch — worked example 3: Add a custom optimizer class to the dispatch registry at runtime

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-class-dispatch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The dispatch dictionary is just a Python dict, so entries can be added or replaced at any time. A `register_optimizer` function that validates the new class and inserts it into the global registry allows plugins or experiments to add optimizers without touching the factory function itself. The factory function continues to work unchanged because it reads from the shared dict.

## Worked solution

**Step 1 — Define the registry with base entries.**
As before, we start with the three standard entries.

**Step 2 — Write `register_optimizer(name, cls)`.**
This function validates that `cls` is a type and a subclass of `torch.optim.Optimizer`, then does `OPTIM_MAP[name] = cls`. Validation at registration time catches mistakes early (rather than at training time when a wrong class would cause a cryptic error).

**Step 3 — Build a minimal custom optimizer.**
We subclass `torch.optim.Optimizer` with a trivial `step` that does standard SGD. The key point is that it IS-A `torch.optim.Optimizer`, so the subclass check passes.

**Step 4 — Register and use it.**
After `register_optimizer('my_sgd', MyCustomSGD)`, calling `build_optimizer('my_sgd', params, lr=0.1)` returns an instance of `MyCustomSGD`. The factory needs no changes.

**Step 5 — Verify rejection of non-optimizer.**
Registering a plain class (not a subclass of Optimizer) raises `TypeError`.

In [ ]:
import torch as t
import torch.nn as nn

OPTIM_MAP = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def register_optimizer(name: str, cls) -> None:
    if not isinstance(cls, type):
        raise TypeError(f'Expected a class, got {type(cls).__name__}')
    if not issubclass(cls, t.optim.Optimizer):
        raise TypeError(f'{cls.__name__} must subclass torch.optim.Optimizer')
    OPTIM_MAP[name] = cls

def build_optimizer(name: str, params, lr: float) -> t.optim.Optimizer:
    return OPTIM_MAP[name](params, lr=lr)

# --- build a tiny custom optimizer ---
class MyCustomSGD(t.optim.Optimizer):
    def __init__(self, params, lr=1e-2):
        super().__init__(params, defaults={'lr': lr})
    @t.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is not None:
                    p -= group['lr'] * p.grad

# --- exercise it ---
t.manual_seed(0)
model = nn.Linear(4, 2)

register_optimizer('my_sgd', MyCustomSGD)
opt = build_optimizer('my_sgd', model.parameters(), lr=0.05)
print(f'type: {type(opt).__name__}')             # MyCustomSGD
print(f'is Optimizer: {isinstance(opt, t.optim.Optimizer)}')  # True
assert isinstance(opt, MyCustomSGD)

# Verify rejection
try:
    register_optimizer('bad', object)
    assert False
except TypeError as e:
    print(f'Correctly rejected: {e}')